In [ ]:
import numpy as np
import pyvista as pv
from pyau3d.files import TopFile, PartFile, ModeFile
from pyau3d.utils import PltFileUtils, NodFileUtils, AnimSmartStream, GrpFileUtils
from pyau3d.pv.loaders import GrpFileVTK
from pyau3d.pv.loader.grpfilevtk import _grp2pv
from pyau3d.pv.vtktool.pdata import polydata
import vtk
import matplotlib.pyplot as plt
from matplotlib import cm, ticker
from pyau3d.constants import Constants
from scipy.signal import hilbert
import time
pv.set_jupyter_backend('trame')  # enables interactive in-notebook rendering
# specify group for animation file
# group = 2 corresponds to cylinder wall - could double check in .an02 file
RE = 60
Mach = 0.2
mesh_ver = 3
GROUP = 1
Mesh = 199560
ianimgrp = GROUP

# cfd_dir = f"/home/ahf25/CFD_2d_cylinder_all/Unsteady/M{Mach}/v{mesh_ver}_mesh/2d_cylinder_{Mesh}_Re{RE}_unsteady"

cfd_dir = f"/mnt/data1/ahf25/run4/2d_cylinder_{Mesh}_Re{RE}_unsteady"
nod_file = f"cylinder.nod"
plt_file = "cylinder.plt"
grp_file =f"cylinder.grp{GROUP:02}"
an_file = f"cylinder.an{GROUP:02}"
part_file = "cylinder.part"
top_file = "cylinder.top"
# specify file paths
path2nod = cfd_dir + "/" + nod_file
path2plt = cfd_dir  + "/" +  plt_file
path2grp = cfd_dir  + "/" +  grp_file
path2an = cfd_dir  + "/" +  an_file
path2part = cfd_dir  + "/" +  part_file
path2top = cfd_dir  + "/" +  top_file
 
mesh = PltFileUtils(path2plt)
nod = NodFileUtils(path2nod)
grp = GrpFileUtils(path2grp,GROUP)
groupfile = GrpFileVTK(path2grp,GROUP)
anim_vars = nod.ivars[GROUP - 1]
part_file = PartFile(path2part)
top = TopFile(path2top)
anim = AnimSmartStream(
    path2an,
    anim_vars=anim_vars,
    part=part_file,
    grp=grp
)
# create surface pv object from group file (file that stores surface geometry)
mesh = groupfile.transformtopv()
# define delta t (for the solver)
deltat = top["dtstr"] / Constants()["u"]

# define delta_t per frame IMPORTANT
frame_rate = 10 # output rate of animation  e.g frame_rate = 10 referes to 1 output per 10 iterations
deltat_frame = deltat * frame_rate # delta t between each frame
# read frames
# define time_selection
start = 1 # start of frame
end = anim.nframes  # end of frame
plot_interval = 50  # internal of this code, does not mean the interval in the animation file
frames_selection = np.arange(start, end, interval)
# ===========================
# = Parameter numbers       =
# ===========================
# Maps parameter number -> (variable name, plot title)
PARAM_MAP = {
    1:  "x-coordinate",
    2:  "y-coordinate",
    3:  "z-coordinate",
    4:  "Density",
    5:  "U-velocity",
    6:  "V-velocity",
    7:  "W-velocity",
    8:  "Internal Energy",
    9:  "Pressure",
    10: "Mach Number",
    11: "Local Work",
}

# ---- CHOOSE YOUR PARAMETER HERE ----
# Set PARAM_NUMBER to a number from PARAM_MAP for a raw variable,
# or set PARAM_NUMBER = "rhou" to view the derived x-momentum field (density * u-velocity)
PARAM_NUMBER = "rhou"
# --------------------------------------

if PARAM_NUMBER == "rhou":
    var_name = "x-momentum (rho*u)"

    # need both density (4) and u-velocity (5) to compute rho*u
    var_selection = np.array([4, 5]) - 1  # 0-indexed

    # read frames: shape (n_frames, n_nodes, 2) -> [:, :, 0]=density, [:, :, 1]=u-velocity
    test = anim.read_frames(frames_selection, var_selection)
    n_frames, n_nodes, n_values = test.shape

    density = test[:, :, 0]
    u_vel = test[:, :, 1]
    scalars = density * u_vel  # rho*u, shape (n_frames, n_nodes)

else:
    var_name = PARAM_MAP[PARAM_NUMBER]
    var_selection = np.array([PARAM_NUMBER]) - 1  # var_selection is 0-index in pyau3d

    # read frames
    test = anim.read_frames(frames_selection, var_selection)
    n_frames, n_nodes, n_values = test.shape

    scalars = test[:, :, 0]  # (n_frames, n_nodes) — scalar case

print("Start of frames: ", start)
print("End of frames: ", end)
print(f"Frame output rate = 1 frame per {frame_rate} iterations")
print(f"Total physical time: {end * deltat_frame}s")
print(f"Viewing parameter: {var_name} (param #{PARAM_NUMBER})")




[SmartStream] Reading Parts: 100%|██████████| 39/39 [01:07<00:00,  1.72s/it]


Start of frames:  1
End of frames:  10000
Interval:  50
Frame output rate = 1 frame per 10 iterations
Total physical time: 1.0s
Viewing parameter: x-momentum (rho*u) (param #rhou)


Widget(value='<iframe src="http://localhost:44595/index.html?ui=P_0x7d8900b8b820_0&reconnect=auto" class="pyvi…

In [5]:
plotter = pv.Plotter(notebook=True)
mesh["field"] = scalars[0]
plotter.add_mesh(mesh, scalars="field", cmap="RdBu",
                  clim=[scalars.min(), scalars.max()], show_scalar_bar=False)

plotter.add_scalar_bar(title=var_name)

def update_frame(frame):
    # map the slider's actual frame number to the nearest available data index
    idx = np.argmin(np.abs(frames_selection - frame))
    mesh["field"] = scalars[idx]
    plotter.render()

plotter.add_slider_widget(
    update_frame,
    rng=[0, end],   # slider now spans the full frame range (0 to 10000)
    value=0,
    title="Frame",
    fmt="%.0f"
)
plotter.view_xy()
plotter.show()

Widget(value='<iframe src="http://localhost:44595/index.html?ui=P_0x7d88ec4d5330_4&reconnect=auto" class="pyvi…

In [7]:
# ===========================
# Export SVG figures for specified frames
# ===========================
export_frames = [0, 250, 500, 750, 1000]  # frame numbers you want to export
output_dir = "svg_frames"  # folder to save the SVGs into

import os
os.makedirs(output_dir, exist_ok=True)

for frame in export_frames:
    # find the nearest available data index for this requested frame
    idx = np.argmin(np.abs(frames_selection - frame))
    actual_frame = frames_selection[idx]

    # create a fresh off-screen plotter for each export (avoids interfering with the interactive widget above)
    export_plotter = pv.Plotter(off_screen=True, notebook=False)
    mesh["field"] = scalars[idx]
    export_plotter.add_mesh(mesh, scalars="field", cmap="RdBu",
                             clim=[scalars.min(), scalars.max()], show_scalar_bar=False)
    export_plotter.add_scalar_bar(title=var_name)

    # add a title showing which frame is being plotted
    export_plotter.add_text(f"{var_name} - Frame {actual_frame}", font_size=14, position='upper_edge')

    export_plotter.view_xy()

    filename = os.path.join(output_dir, f"frame_{frame:05d}.svg")
    export_plotter.save_graphic(filename)
    export_plotter.close()

    print(f"Saved frame {frame} (actual data frame {actual_frame}) -> {filename}")

print(f"\nDone. {len(export_frames)} SVG files saved to '{output_dir}/'")

Saved frame 0 (actual data frame 1) -> svg_frames/frame_00000.svg
Saved frame 250 (actual data frame 251) -> svg_frames/frame_00250.svg
Saved frame 500 (actual data frame 501) -> svg_frames/frame_00500.svg
Saved frame 750 (actual data frame 751) -> svg_frames/frame_00750.svg
Saved frame 1000 (actual data frame 1001) -> svg_frames/frame_01000.svg

Done. 5 SVG files saved to 'svg_frames/'
